# Factor Model Exploration

Scratch notebook for data exploration and prototyping before implementing `src/data_loader.py`.

**Goal:** Understand the WRDS data schemas, column names, and date coverage for CRSP and Fama-French — so we can write correct, targeted queries in the production loader.

**Sections:**
1. Setup & WRDS connection
2. CRSP monthly stock data
3. Fama-French 5-factor data
4. CRSP names table
5. Data coverage checks

## 1. Setup & WRDS Connection

WRDS uses PostgreSQL under the hood. The `connect_wrds()` function reads `WRDS_USERNAME` from `.env` and prompts for your password on first use — credentials are then cached at `~/.pgpass` for future sessions.

> **Note:** `wrds 3.x` and `pandas 2.x` have a known incompatibility with `conn.raw_sql()`. We use `pd.read_sql(sql, conn.engine)` directly instead, which bypasses the issue.

In [1]:
import sys
sys.path.insert(0, '..')  # resolve src/ imports from notebooks/ subdirectory

import pandas as pd
import numpy as np
from src.config import *
from src.data_loader import connect_wrds

conn = connect_wrds()  # prompts for password on first use, then caches to ~/.pgpass

def query(sql):
    """Query WRDS using raw psycopg2, bypassing pandas/SQLAlchemy version conflicts.

    wrds pins pandas<2.3 and uses SQLAlchemy 1.4; pandas 2.3 no longer recognises
    SQLAlchemy 1.4 objects. Dropping down to the underlying psycopg2 cursor avoids
    the entire compatibility layer.
    """
    pgconn = conn.connection.connection
    with pgconn.cursor() as cur:
        cur.execute(sql)
        cols = [d[0] for d in cur.description]
        df = pd.DataFrame(cur.fetchall(), columns=cols)
    # coerce columns to numeric where possible, leave strings (e.g. ticker) as-is
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='ignore')
    return df

Loading library list...
Done


## 2. CRSP Monthly Stock Data

CRSP (Center for Research in Security Prices) is the primary source for US equity returns. The key table is `crsp.msf` — the **monthly stock file**.

Key columns in `crsp.msf`:

| Column | Description |
|--------|-------------|
| `permno` | Permanent stock identifier (stable across ticker changes) |
| `date` | Month-end date |
| `ret` | Monthly return (decimal, e.g. 0.05 = 5%) |
| `prc` | Price (negative = average of bid/ask when closing price unavailable) |
| `shrout` | Shares outstanding (thousands) — multiply by `abs(prc)` for market cap |

> `exchcd` (exchange code) and `shrcd` (share code) are **not** in `msf` — they live in `crsp.msenames` and must be joined on `permno` using the `namedt`/`nameendt` date range.

In [2]:
# List all tables in the CRSP library
conn.list_tables(library='crsp')

['acti',
 'asia',
 'asib',
 'asic',
 'asio',
 'asix',
 'bmdebt',
 'bmheader',
 'bmpaymts',
 'bmquotes',
 'bmyield',
 'bndprt06',
 'bndprt12',
 'bxcalind',
 'bxdlyind',
 'bxmthind',
 'bxquotes',
 'bxyield',
 'cap',
 'ccm_lookup',
 'ccm_qvards',
 'ccmxpf_linktable',
 'ccmxpf_lnkhist',
 'ccmxpf_lnkrng',
 'ccmxpf_lnkused',
 'comphead',
 'comphist',
 'compmaster',
 'contact_info',
 'core',
 'crsp_cik_map',
 'crsp_daily_data',
 'crsp_header',
 'crsp_monthly_data',
 'crsp_names',
 'crsp_portno_map',
 'crsp_ziman_daily_index',
 'crsp_ziman_monthly_index',
 'cs20yr',
 'cs5yr',
 'cs90d',
 'cst_hist',
 'daily_nav',
 'daily_nav_ret',
 'daily_returns',
 'dividends',
 'dport1',
 'dport2',
 'dport3',
 'dport4',
 'dport5',
 'dport6',
 'dport7',
 'dport8',
 'dport9',
 'dsbc',
 'dsbo',
 'dse',
 'dse62',
 'dse62delist',
 'dse62dist',
 'dse62exchdates',
 'dse62names',
 'dse62nasdin',
 'dse62shares',
 'dseall',
 'dseall62',
 'dsedelist',
 'dsedist',
 'dseexchdates',
 'dsenames',
 'dsenasdin',
 'dseshares',

In [3]:
# Inspect CRSP monthly stock file schema
conn.describe_table(library='crsp', table='msf')

Approximately 5153763 rows in crsp.msf.


,name,nullable,type,comment
0,cusip,True,VARCHAR(8),CUSIP Header
1,permno,True,INTEGER,PERMNO
2,permco,True,INTEGER,PERMCO
3,issuno,True,INTEGER,Nasdaq Issue Number
4,hexcd,True,SMALLINT,Exchange Code Header
5,hsiccd,True,INTEGER,Standard Industrial Classification Code Header
6,date,True,DATE,Date of Observation
7,bidlo,True,"NUMERIC(11, 5)",Bid or Low Price
8,askhi,True,"NUMERIC(11, 5)",Ask or High Price
9,prc,True,"NUMERIC(11, 5)",Price or Bid/Ask Average


In [4]:
# Pull a small CRSP sample to inspect data shape and values
# exchcd and shrcd are in msenames, not msf — join to get them
crsp_sample = query("""
    SELECT m.permno, m.date, m.ret, m.prc, m.shrout,
           n.exchcd, n.shrcd, n.ticker, n.comnam
    FROM crsp.msf m
    LEFT JOIN crsp.msenames n
        ON m.permno = n.permno
        AND m.date BETWEEN n.namedt AND n.nameendt
    WHERE m.date BETWEEN '2020-01-01' AND '2020-12-31'
    LIMIT 1000
""")
crsp_sample.head()

/var/folders/sm/qkbwm5vj7s53y7x_c_4lhv7w0000gn/T/ipykernel_76025/588027662.py:25: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


,permno,date,ret,prc,shrout,exchcd,shrcd,ticker,comnam
0,10026,2020-01-31,-0.100016,165.84,18919.0,3.0,11.0,JJSF,J & J SNACK FOODS CORP
1,10028,2020-01-31,0.607407,2.17,26924.0,2.0,11.0,ELA,ENVELA CORP
2,10032,2020-01-31,-0.075643,71.12,29222.0,3.0,11.0,PLXS,PLEXUS CORP
3,10044,2020-01-31,-0.098592,8.32,6000.0,3.0,11.0,RMCF,ROCKY MOUNTAIN CHOC FAC INC NEW
4,10051,2020-01-31,-0.115176,24.43,37338.0,1.0,11.0,HNGR,HANGER INC


## 3. Fama-French 5-Factor Data

The FF5 factors are pulled from `ff.factors_monthly`. These are the right-hand-side variables in our OLS regressions.

> **Note:** WRDS stores factor names in lowercase (`mktrf`, `smb`, `hml`, `rmw`, `cma`, `rf`). Our `config.FACTORS` list uses the canonical capitalized names — we'll need to map these when merging.

In [5]:
# List all tables in the Fama-French library
conn.list_tables(library='ff')

['factors_china',
 'factors_daily',
 'factors_monthly',
 'fivefactors_daily',
 'fivefactors_monthly',
 'industry12',
 'industry48',
 'liq_ps',
 'liq_sadka',
 'portfolios',
 'portfolios25',
 'portfolios_d']

In [6]:
# Pull Fama-French 5-factor monthly data sample
# Column names: date, mktrf, smb, hml, rmw, cma, rf
ff_sample = query("""
    SELECT *
    FROM ff.factors_monthly
    WHERE date BETWEEN '2020-01-01' AND '2020-12-31'
""")
ff_sample.head(20)

/var/folders/sm/qkbwm5vj7s53y7x_c_4lhv7w0000gn/T/ipykernel_76025/588027662.py:25: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


,date,mktrf,smb,hml,rf,year,month,umd,dateff
0,2020-01-01,-0.0011,-0.0310,-0.0622,0.0013,2020.0,1.0,0.0602,2020-01-31
1,2020-02-01,-0.0815,0.0108,-0.0382,0.0012,2020.0,2.0,-0.0028,2020-02-28
2,2020-03-01,-0.1337,-0.0469,-0.1383,0.0012,2020.0,3.0,0.0821,2020-03-31
3,2020-04-01,0.1360,0.0250,-0.0134,0.0000,2020.0,4.0,-0.0548,2020-04-30
4,2020-05-01,0.0557,0.0240,-0.0500,0.0001,2020.0,5.0,0.0016,2020-05-29
5,2020-06-01,0.0245,0.0267,-0.0217,0.0001,2020.0,6.0,-0.0083,2020-06-30
6,2020-07-01,0.0583,-0.0235,-0.0144,0.0001,2020.0,7.0,0.0771,2020-07-31
7,2020-08-01,0.0762,-0.0020,-0.0302,0.0001,2020.0,8.0,0.0046,2020-08-31
8,2020-09-01,-0.0364,0.0006,-0.0272,0.0001,2020.0,9.0,0.0319,2020-09-30
9,2020-10-01,-0.0208,0.0434,0.0425,0.0001,2020.0,10.0,-0.0314,2020-10-30


## 4. CRSP Names Table

`crsp.msenames` maps `permno` to human-readable identifiers (ticker, company name). A single `permno` can have multiple rows as tickers and names change over time — use `namedt`/`nameendt` to get the active name at a given date.

We'll use this to let users search for stocks by ticker in the Streamlit app.

In [7]:
# Inspect names table schema
conn.describe_table(library='crsp', table='msenames')

Approximately 117830 rows in crsp.msenames.


,name,nullable,type,comment
0,permno,True,INTEGER,PERMNO
1,namedt,True,DATE,Names Date
2,nameendt,True,DATE,Names Ending Date
3,shrcd,True,SMALLINT,Share Code
4,exchcd,True,SMALLINT,Exchange Code
5,siccd,True,INTEGER,Standard Industrial Classification Code
6,ncusip,True,VARCHAR(8),CUSIP
7,ticker,True,VARCHAR(8),Ticker Symbol
8,comnam,True,VARCHAR(35),Company Name
9,shrcls,True,VARCHAR(4),Share Class


In [8]:
# Sample the names table
names_sample = query("""
    SELECT permno, comnam, ticker, namedt, nameendt, exchcd, shrcd
    FROM crsp.msenames
    LIMIT 100
""")
names_sample.head()

/var/folders/sm/qkbwm5vj7s53y7x_c_4lhv7w0000gn/T/ipykernel_76025/588027662.py:25: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


,permno,comnam,ticker,namedt,nameendt,exchcd,shrcd
0,10000,OPTIMUM MANUFACTURING INC,OMFGA,1986-01-07,1986-12-03,3,10
1,10000,OPTIMUM MANUFACTURING INC,OMFGA,1986-12-04,1987-03-09,3,10
2,10000,OPTIMUM MANUFACTURING INC,OMFGA,1987-03-10,1987-06-11,3,10
3,10001,GREAT FALLS GAS CO,GFGC,1986-01-09,1993-11-21,3,11
4,10001,ENERGY WEST INC,EWST,1993-11-22,2004-06-09,3,11


## 5. Data Coverage Checks

Verify that the data spans our target range (`START_DATE` to `END_DATE` from `config.py`) before committing to those constants in the production loader.

In [9]:
# Check full date range and row count in CRSP monthly file
query("SELECT MIN(date), MAX(date), COUNT(*) FROM crsp.msf")

/var/folders/sm/qkbwm5vj7s53y7x_c_4lhv7w0000gn/T/ipykernel_76025/588027662.py:25: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


,min,max,count
0,1925-12-31,2024-12-31,5153763


---

## 6. Fama-French 3-Factor Decomposition: Apple (AAPL)

The FF3 model decomposes a stock's excess return into three systematic factors plus an idiosyncratic residual:

$$R_i - R_f = \alpha + \beta_{MKT}(R_m - R_f) + \beta_{SMB} \cdot SMB + \beta_{HML} \cdot HML + \varepsilon$$

| Term | Meaning |
|------|---------|
| $R_i - R_f$ | Stock excess return (return minus risk-free rate) |
| $\alpha$ | Idiosyncratic return (Jensen's alpha) — return unexplained by factors |
| $\beta_{MKT}$ | Market beta — sensitivity to broad market moves |
| $\beta_{SMB}$ | Size beta — tilt toward small vs. large caps |
| $\beta_{HML}$ | Value beta — tilt toward value vs. growth stocks |
| $\varepsilon$ | Residual — idiosyncratic noise |

In [10]:
# Step 1: Find Apple's PERMNO
apple_permno = query("""
    SELECT permno, comnam, ticker, namedt, nameendt
    FROM crsp.msenames
    WHERE ticker = 'AAPL'
    ORDER BY namedt DESC
    LIMIT 5
""")
apple_permno

/var/folders/sm/qkbwm5vj7s53y7x_c_4lhv7w0000gn/T/ipykernel_76025/588027662.py:25: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


,permno,comnam,ticker,namedt,nameendt
0,14593,APPLE INC,AAPL,2017-12-28,2024-12-31
1,14593,APPLE INC,AAPL,2007-01-11,2017-12-27
2,14593,APPLE COMPUTER INC,AAPL,2004-06-10,2007-01-10
3,14593,APPLE COMPUTER INC,AAPL,1982-11-01,2004-06-09
4,14593,APPLE COMPUTER INC,AAPL,1980-12-12,1982-10-31


In [11]:
# Step 2: Pull 5 years of Apple monthly returns (2019-2023)
permno = apple_permno['permno'].iloc[0]

aapl_returns = query(f"""
    SELECT date, ret, prc, shrout
    FROM crsp.msf
    WHERE permno = {permno}
      AND date BETWEEN '2019-01-01' AND '2023-12-31'
    ORDER BY date
""")

aapl_returns['date'] = pd.to_datetime(aapl_returns['date'])
aapl_returns = aapl_returns.dropna(subset=['ret'])
print(f"Observations: {len(aapl_returns)}")
aapl_returns.head()

Observations: 60


/var/folders/sm/qkbwm5vj7s53y7x_c_4lhv7w0000gn/T/ipykernel_76025/588027662.py:25: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


,date,ret,prc,shrout
0,2019-01-31,0.055154,166.44000,4715280.0
1,2019-02-28,0.044701,173.14999,4715280.0
2,2019-03-29,0.097026,189.95000,4715280.0
3,2019-04-30,0.056436,200.67000,4601075.0
4,2019-05-31,-0.123735,175.07001,4601075.0


In [12]:
# Step 3: Pull FF3 factors for the same period
# rf is the monthly risk-free rate; mktrf is already the excess market return
ff3 = query("""
    SELECT date, mktrf, smb, hml, rf
    FROM ff.factors_monthly
    WHERE date BETWEEN '2019-01-01' AND '2023-12-31'
    ORDER BY date
""")

ff3['date'] = pd.to_datetime(ff3['date'])
ff3.head()

/var/folders/sm/qkbwm5vj7s53y7x_c_4lhv7w0000gn/T/ipykernel_76025/588027662.py:25: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


,date,mktrf,smb,hml,rf
0,2019-01-01,0.0837,0.0276,-0.0039,0.0021
1,2019-02-01,0.0342,0.0205,-0.0266,0.0018
2,2019-03-01,0.0110,-0.0303,-0.0414,0.0019
3,2019-04-01,0.0397,-0.0174,0.0213,0.0021
4,2019-05-01,-0.0692,-0.0126,-0.0245,0.0021


In [13]:
# Step 4: Merge and compute excess return
# CRSP dates = last trading day of month; FF dates = last calendar day
# Normalize both to year-month period to align them
df = aapl_returns.copy()
df['ym'] = df['date'].dt.to_period('M')
ff3['ym'] = ff3['date'].dt.to_period('M')

df = df.merge(ff3.drop(columns='date'), on='ym', how='inner')
df['excess_ret'] = df['ret'] - df['rf']

print(f"Merged observations: {len(df)}")
df[['date', 'ret', 'rf', 'excess_ret', 'mktrf', 'smb', 'hml']].head()

Merged observations: 60


,date,ret,rf,excess_ret,mktrf,smb,hml
0,2019-01-31,0.055154,0.0021,0.053054,0.0837,0.0276,-0.0039
1,2019-02-28,0.044701,0.0018,0.042901,0.0342,0.0205,-0.0266
2,2019-03-29,0.097026,0.0019,0.095126,0.0110,-0.0303,-0.0414
3,2019-04-30,0.056436,0.0021,0.054336,0.0397,-0.0174,0.0213
4,2019-05-31,-0.123735,0.0021,-0.125835,-0.0692,-0.0126,-0.0245


In [14]:
# Step 5: OLS regression — FF3 model
import statsmodels.api as sm

# psycopg2 returns columns as object dtype — cast to float explicitly
cols = ['excess_ret', 'mktrf', 'smb', 'hml']
df[cols] = df[cols].astype(float)

X = sm.add_constant(df[['mktrf', 'smb', 'hml']])
y = df['excess_ret']

result = sm.OLS(y, X).fit()
print(result.summary())

                            OLS Regression Results                            
Dep. Variable:             excess_ret   R-squared:                       0.720
Model:                            OLS   Adj. R-squared:                  0.705
Method:                 Least Squares   F-statistic:                     48.10
Date:                Sat, 28 Mar 2026   Prob (F-statistic):           1.64e-15
Time:                        17:05:40   Log-Likelihood:                 100.65
No. Observations:                  60   AIC:                            -193.3
Df Residuals:                      56   BIC:                            -184.9
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0133      0.006      2.137      0.0

In [15]:
# Step 6: Print the fitted equation
alpha  = result.params['const']
b_mkt  = result.params['mktrf']
b_smb  = result.params['smb']
b_hml  = result.params['hml']
r2     = result.rsquared

print("Fitted Fama-French 3-Factor Model for AAPL (2019–2023)")
print("=" * 55)
print(f"  R - Rf = {alpha:+.4f} + {b_mkt:.4f}·MKT + {b_smb:.4f}·SMB + {b_hml:.4f}·HML + ε")
print()
print(f"  α  (idio/alpha) : {alpha:+.4f}  (p={result.pvalues['const']:.3f})")
print(f"  β  (market)     : {b_mkt:.4f}  (p={result.pvalues['mktrf']:.3f})")
print(f"  s  (SMB)        : {b_smb:.4f}  (p={result.pvalues['smb']:.3f})")
print(f"  h  (HML)        : {b_hml:.4f}  (p={result.pvalues['hml']:.3f})")
print()
print(f"  R²              : {r2:.4f}")
print(f"  Idio variance   : {1 - r2:.4f}  ({(1-r2)*100:.1f}% unexplained by factors)")

Fitted Fama-French 3-Factor Model for AAPL (2019–2023)
  R - Rf = +0.0133 + 1.3302·MKT + -0.4276·SMB + -0.4804·HML + ε

  α  (idio/alpha) : +0.0133  (p=0.037)
  β  (market)     : 1.3302  (p=0.000)
  s  (SMB)        : -0.4276  (p=0.060)
  h  (HML)        : -0.4804  (p=0.000)

  R²              : 0.7204
  Idio variance   : 0.2796  (28.0% unexplained by factors)


---

## 7. FF3 Function + Multi-Stock Comparison

Generalise the Apple analysis into a reusable function, then compare AAPL, CVX, and WMT.

In [16]:
import statsmodels.api as sm

def ff3_decompose(ticker, start='2019-01-01', end='2023-12-31'):
    """Run FF3 regression for a ticker and return coefficients + monthly contributions.

    Returns
    -------
    result : statsmodels RegressionResults
    contrib : DataFrame with columns [date, excess_ret, alpha, mkt, smb, hml, residual]
    """
    # 1. Look up permno
    names = query(f"""
        SELECT permno FROM crsp.msenames
        WHERE ticker = '{ticker}'
        ORDER BY namedt DESC LIMIT 1
    """)
    if names.empty:
        raise ValueError(f"Ticker {ticker} not found in CRSP")
    permno = int(names['permno'].iloc[0])

    # 2. Pull monthly returns
    returns = query(f"""
        SELECT date, ret FROM crsp.msf
        WHERE permno = {permno}
          AND date BETWEEN '{start}' AND '{end}'
        ORDER BY date
    """)
    returns['date'] = pd.to_datetime(returns['date'])
    returns = returns.dropna(subset=['ret'])
    returns['ym'] = returns['date'].dt.to_period('M')

    # 3. Pull FF3 factors
    factors = query(f"""
        SELECT date, mktrf, smb, hml, rf FROM ff.factors_monthly
        WHERE date BETWEEN '{start}' AND '{end}'
        ORDER BY date
    """)
    factors['date'] = pd.to_datetime(factors['date'])
    factors['ym'] = factors['date'].dt.to_period('M')

    # 4. Merge and compute excess return
    df = returns.merge(factors.drop(columns='date'), on='ym', how='inner')
    num_cols = ['ret', 'rf', 'mktrf', 'smb', 'hml']
    df[num_cols] = df[num_cols].astype(float)
    df['excess_ret'] = df['ret'] - df['rf']

    # 5. OLS regression
    X = sm.add_constant(df[['mktrf', 'smb', 'hml']])
    result = sm.OLS(df['excess_ret'], X).fit()

    # 6. Monthly factor contributions
    a     = result.params['const']
    b_mkt = result.params['mktrf']
    b_smb = result.params['smb']
    b_hml = result.params['hml']

    contrib = df[['date', 'excess_ret']].copy()
    contrib['alpha']    = a                        # constant each month
    contrib['mkt']      = b_mkt * df['mktrf']     # market contribution
    contrib['smb']      = b_smb * df['smb']       # size contribution
    contrib['hml']      = b_hml * df['hml']       # value contribution
    contrib['residual'] = (df['excess_ret']
                           - a
                           - contrib['mkt']
                           - contrib['smb']
                           - contrib['hml'])

    print(f"\n{'='*52}")
    print(f"  FF3 — {ticker}  ({start} to {end})")
    print(f"{'='*52}")
    print(f"  R - Rf = {a:+.4f} "
          f"+ {b_mkt:.4f}·MKT "
          f"+ {b_smb:.4f}·SMB "
          f"+ {b_hml:.4f}·HML + ε")
    print(f"  R² = {result.rsquared:.4f}   "
          f"Idio variance = {1-result.rsquared:.4f}")

    return result, contrib

In [20]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

tickers  = ['AAPL', 'CVX', 'WMT']
contribs = [contrib_aapl, contrib_cvx, contrib_wmt]

COMPONENTS = {
    'alpha':      ('Alpha (idio)',        '#636EFA'),
    'mkt':        ('β · MKT',            '#EF553B'),
    'smb':        ('s · SMB',            '#00CC96'),
    'hml':        ('h · HML',            '#AB63FA'),
    'residual':   ('Residual (ε)',        '#FFA15A'),
    'excess_ret': ('Total excess return', '#000000'),
}

LINE_STYLES = {
    'alpha':      dict(width=1.5),
    'mkt':        dict(width=1.5),
    'smb':        dict(width=1.5),
    'hml':        dict(width=1.5),
    'residual':   dict(width=1.5, dash='dot'),
    'excess_ret': dict(width=2.2,  dash='dash'),
}

fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    subplot_titles=tickers,
    vertical_spacing=0.08,
)

for row, (ticker, contrib) in enumerate(zip(tickers, contribs), start=1):
    for col, (label, color) in COMPONENTS.items():
        fig.add_trace(
            go.Scatter(
                x=contrib['date'],
                y=contrib[col],
                name=label,
                legendgroup=label,
                showlegend=(row == 1),
                line=dict(color=color, **LINE_STYLES[col]),
                hovertemplate=f'<b>{label}</b>: %{{y:.2%}}<extra></extra>',
            ),
            row=row, col=1,
        )

fig.update_layout(
    height=800,
    title_text='Fama-French 3-Factor Return Decomposition (2019–2023)',
    hovermode='x unified',
    legend=dict(orientation='h', y=-0.08),
)
fig.update_yaxes(tickformat='.0%', title_text='Monthly return')
fig.show()

---

## 8. Equal-Weight Portfolio: Factor Decomposition & Market Hedge

**Workflow:**
1. Build an equal-weight portfolio of AAPL, CVX, WMT
2. Run FF3 regression → get portfolio equation
3. Short `β_p` units of the market (S&P proxy) to zero out market risk
4. Re-run regression on the hedged portfolio → confirm β ≈ 0
5. Decompose portfolio variance by source: MKT, SMB, HML, idiosyncratic

> **Note on alpha vs idiosyncratic risk:** The *alpha* (α) in the equation is a constant — it tells you about *return*, not *risk*. *Idiosyncratic risk* is the variance of the residual ε — the part of return volatility unexplained by any factor. These are related concepts but not the same thing.

In [27]:
# Step 1: Build equal-weight portfolio excess returns
# Align all three contrib DataFrames on date, then average excess returns
port = (contrib_aapl[['date', 'excess_ret']].rename(columns={'excess_ret': 'aapl'})
        .merge(contrib_cvx[['date', 'excess_ret']].rename(columns={'excess_ret': 'cvx'}),  on='date')
        .merge(contrib_wmt[['date', 'excess_ret']].rename(columns={'excess_ret': 'wmt'}),  on='date'))

port['port_ret'] = (port['aapl'] + port['cvx'] + port['wmt']) / 3

# Also pull factor returns aligned to these dates
factors_port = ff3.copy()  # already in memory from earlier cells
factors_port['ym'] = factors_port['date'].dt.to_period('M')
port['ym'] = port['date'].dt.to_period('M')
port = port.merge(factors_port[['ym', 'mktrf', 'smb', 'hml']], on='ym', how='inner')

print(f"Portfolio observations: {len(port)}")
port[['date', 'aapl', 'cvx', 'wmt', 'port_ret']].head()

Portfolio observations: 60


,date,aapl,cvx,wmt,port_ret
0,2019-01-31,0.053054,0.051765,0.026671,0.043830
1,2019-02-28,0.042901,0.051580,0.031175,0.041885
2,2019-03-29,0.095126,0.028205,-0.011295,0.037345
3,2019-04-30,0.054336,-0.027429,0.052345,0.026417
4,2019-05-31,-0.125835,-0.043912,-0.010560,-0.060102


In [28]:
# Step 2: FF3 regression on the unhedged portfolio
X = sm.add_constant(port[['mktrf', 'smb', 'hml']])
res_port = sm.OLS(port['port_ret'], X).fit()

a_p     = res_port.params['const']
b_mkt_p = res_port.params['mktrf']
b_smb_p = res_port.params['smb']
b_hml_p = res_port.params['hml']

print("Equal-Weight Portfolio — FF3 Equation")
print("=" * 52)
print(f"  R_p - Rf = {a_p:+.4f} + {b_mkt_p:.4f}·MKT + {b_smb_p:.4f}·SMB + {b_hml_p:.4f}·HML + ε")
print(f"  R² = {res_port.rsquared:.4f}")
print()
print(f"  To zero market risk: short {b_mkt_p:.4f} units of the market per 1 unit of portfolio")

Equal-Weight Portfolio — FF3 Equation
  R_p - Rf = +0.0055 + 0.9700·MKT + -0.3215·SMB + 0.0884·HML + ε
  R² = 0.7731

  To zero market risk: short 0.9700 units of the market per 1 unit of portfolio


In [29]:
# Step 3: Hedge — short β_p units of the market to zero out market risk
# Hedged return = portfolio return - β_p * MKT
port['hedged_ret'] = port['port_ret'] - b_mkt_p * port['mktrf']

# Step 4: Re-run FF3 on hedged portfolio
X_h = sm.add_constant(port[['mktrf', 'smb', 'hml']])
res_hedged = sm.OLS(port['hedged_ret'], X_h).fit()

a_h     = res_hedged.params['const']
b_mkt_h = res_hedged.params['mktrf']
b_smb_h = res_hedged.params['smb']
b_hml_h = res_hedged.params['hml']

print("Hedged Portfolio (β_MKT ≈ 0) — FF3 Equation")
print("=" * 52)
print(f"  R_h - Rf = {a_h:+.4f} + {b_mkt_h:.4f}·MKT + {b_smb_h:.4f}·SMB + {b_hml_h:.4f}·HML + ε")
print(f"  R² = {res_hedged.rsquared:.4f}")
print()
print(f"  Market beta: {b_mkt_h:.6f}  (≈ 0 ✓)" if abs(b_mkt_h) < 0.01 else f"  Market beta: {b_mkt_h:.6f}")

Hedged Portfolio (β_MKT ≈ 0) — FF3 Equation
  R_h - Rf = +0.0055 + 0.0000·MKT + -0.3215·SMB + 0.0884·HML + ε
  R² = 0.1121

  Market beta: 0.000000  (≈ 0 ✓)


In [30]:
# Step 5: Variance decomposition
# Assuming near-orthogonal factors, each factor's variance contribution = β²·Var(factor)
# Idiosyncratic = residual variance

def variance_decomposition(result, factors_df):
    """Break total return variance into factor contributions + idiosyncratic."""
    b_mkt = result.params['mktrf']
    b_smb = result.params['smb']
    b_hml = result.params['hml']

    var_mkt  = (b_mkt ** 2) * factors_df['mktrf'].var()
    var_smb  = (b_smb ** 2) * factors_df['smb'].var()
    var_hml  = (b_hml ** 2) * factors_df['hml'].var()
    var_idio = result.resid.var()
    total    = var_mkt + var_smb + var_hml + var_idio

    return {
        'MKT risk':           var_mkt  / total,
        'SMB risk':           var_smb  / total,
        'HML risk':           var_hml  / total,
        'Idiosyncratic (ε)':  var_idio / total,
    }

decomp_unhedged = variance_decomposition(res_port,   port)
decomp_hedged   = variance_decomposition(res_hedged, port)

print("Variance Decomposition — Unhedged Portfolio")
print("-" * 42)
for k, v in decomp_unhedged.items():
    print(f"  {k:<24} {v:>6.1%}")

print()
print("Variance Decomposition — Hedged Portfolio (β_MKT = 0)")
print("-" * 42)
for k, v in decomp_hedged.items():
    print(f"  {k:<24} {v:>6.1%}")

Variance Decomposition — Unhedged Portfolio
------------------------------------------
  MKT risk                  76.4%
  SMB risk                   2.3%
  HML risk                   0.5%
  Idiosyncratic (ε)         20.8%

Variance Decomposition — Hedged Portfolio (β_MKT = 0)
------------------------------------------
  MKT risk                   0.0%
  SMB risk                   9.7%
  HML risk                   1.9%
  Idiosyncratic (ε)         88.4%


In [37]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

STACK_COMPONENTS = {
    'alpha':    ('Alpha (idio)', '#636EFA'),
    'mkt':      ('β · MKT',     '#EF553B'),
    'smb':      ('s · SMB',     '#00CC96'),
    'hml':      ('h · HML',     '#AB63FA'),
    'residual': ('Residual (ε)','#FFA15A'),
}

fig = make_subplots(
    rows=3, cols=2,
    specs=[
        [{'type': 'xy', 'colspan': 2}, None],
        [{'type': 'xy', 'colspan': 2}, None],
        [{'type': 'pie'}, {'type': 'pie'}],
    ],
    vertical_spacing=0.12,
    row_heights=[0.34, 0.34, 0.32],
)

for row, (contrib, stack_id) in enumerate([
    (port_contrib,   'stack1'),
    (hedged_contrib, 'stack2'),
], start=1):
    for key, (label, color) in STACK_COMPONENTS.items():
        fig.add_trace(go.Scatter(
            x=contrib['date'],
            y=contrib[key],
            name=label,
            legendgroup=label,
            showlegend=(row == 1),
            stackgroup=stack_id,
            fillcolor=color,
            line=dict(color=color, width=0.5),
            hovertemplate=f'<b>{label}</b>: %{{y:.2%}}<extra></extra>',
        ), row=row, col=1)

    fig.add_trace(go.Scatter(
        x=contrib['date'],
        y=contrib['total_ret'],
        name='Total return',
        legendgroup='Total return',
        showlegend=(row == 1),
        line=dict(color='black', width=2, dash='dash'),
        hovertemplate='<b>Total return</b>: %{y:.2%}<extra></extra>',
    ), row=row, col=1)

for col, decomp in [(1, decomp_unhedged), (2, decomp_hedged)]:
    fig.add_trace(go.Pie(
        labels=list(decomp.keys()),
        values=list(decomp.values()),
        textinfo='label+percent',
        textfont_size=11,
        showlegend=False,
        marker_colors=['#EF553B', '#00CC96', '#AB63FA', '#FFA15A'],
        hole=0.35,
    ), row=3, col=col)

fig.update_layout(
    height=980,
    title_text='Equal-Weight Portfolio: Return Decomposition & Risk Attribution',
    hovermode='x unified',
    legend=dict(orientation='h', x=0, y=1.06, xanchor='left'),
    margin=dict(b=80),
    annotations=[
        dict(text='Unhedged Portfolio',          x=0.5,  y=1.01,  xref='paper', yref='paper', showarrow=False, font=dict(size=13)),
        dict(text='Hedged Portfolio (β_MKT=0)',  x=0.5,  y=0.63,  xref='paper', yref='paper', showarrow=False, font=dict(size=13)),
        dict(text='Risk Attribution — Unhedged', x=0.25, y=-0.06, xref='paper', yref='paper', showarrow=False, font=dict(size=12)),
        dict(text='Risk Attribution — Hedged',   x=0.75, y=-0.06, xref='paper', yref='paper', showarrow=False, font=dict(size=12)),
    ],
)
fig.update_yaxes(tickformat='.0%')
fig.show()